# Imputation Benchmark: Method Comparison (Pseudo-6K Panel)

Evaluates six spatial gene imputation methods on a held-out CosMx WTX (whole-transcriptome) dataset.

**Benchmark design (pseudo-6K)**:  
Cells from held-out FOVs are masked to the 6K-gene panel; methods impute the remaining ~12K genes.  
Performance is measured against the true WTX expression.

**Methods**: GCN (our model), Tangram, gimVI, stPlus, SPRITE, CellPLM

**Metrics**:
- Cell-type classification: Hamming loss and Micro F1 (via AUCell re-annotation)
- Clustering quality: Silhouette, Calinski-Harabasz, Davies-Bouldin (PCA + Harmony)
- Per-gene correlation: Pearson (PCC) and Spearman (SCC)

**Inputs**: Pre-computed imputed count CSVs and AUCell annotation CSVs (produced by individual method notebooks)


In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import hamming_loss, f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.stats import pearsonr, spearmanr

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cosmx/data"
AUCELL_DIR = "/path/to/cosmx/aucell"
COUNT_DIR  = "/path/to/cosmx/count_csv"
MAPPER_CSV = "/path/to/scrna/cell_type_name_mapper.csv"

# Cell type display order (fine annotation)
cell_type_order = [
    'Cycl T', 'Resting T', 'DN T', 'Gd T', 'Gd T (Vd2g9+)', 'MAIT',
    'CD4 Trm', 'CD4 Effector', 'Treg', 'Tfh',
    'GZMK- CD8 Trm', 'GZMK+ CD8 Trm', 'CD8 Effector', 'NK',
    'B cell', 'Plasma',
    'BEST4+ cell', 'Inflamed Epi', 'Enteroendocrine', 'M cell', 'Tuft',
    'Enterocyte', 'Imm Enterocyte', 'Secre Progenitor', 'Goblet', 'Imm Goblet',
    'Absorp Progenitor', 'TA1', 'TA2', 'Stem', 'Cycl Epi',
    'NRG1+ Crypt Top Fib', 'VSTM2A+ Crypt Top Fib', 'OGN+RSPO3+ Fib',
    'SELENOP+ Fib', 'ADAMDEC1+ Fib', 'Activ Fib', 'CXCL5+ Activ Fib',
    'T-interact Fib', 'Cycl Fib', 'HHIP+ Myofib', 'GREM2+ Myofib',
    'RERGL+ Contr Peri', 'CD36+ Peri', 'Endothelial', 'Neuron',
    'Neutrophil', 'cDC1', 'CCR7+ DC', 'cDC2', 'Trans cDC2/Mac',
    'Mac S+SG+', 'Mac S+XS-', 'Mac S+M+S+', 'Mac S+M+P+',
    'Mo-Mac', 'Inf Mo-Mac', 'Cycl Myeloid', 'Granulocyte'
]

mapper = pd.read_csv(MAPPER_CSV)
mapper_sub = mapper[['cell_type_short', 'cell_category']]

## 1. Load WTX data and define test set

In [ ]:
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_I0261_I0331_anno.h5ad")
adata.obs['fov'] = adata.obs['fov'].astype(int)
adata.obs_names = adata.obs['fov_cell_id'].tolist()

# Train/test split by FOV: right tissue = train, left tissue = test
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs.index):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)
print(f"Train: {len(train_cells):,}  |  Test: {len(test_cells):,}")

adata_test = adata[test_cells, :].copy()

# 6K gene panel (the observed genes in the pseudo-6K experiment)
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()

# Ground truth expression matrix (test cells, 6K genes only)
df_true = pd.DataFrame(adata_test.X)
df_true.columns = adata_test.var_names.tolist()
df_true['fov_cell_id'] = adata_test.obs['fov_cell_id'].tolist()
df_true = df_true.set_index('fov_cell_id')
df_true_sub = df_true[genes_6k]

## 2. AUCell cell type annotations per method

In [ ]:
def load_aucell(csv_path, short_col, full_col, cat_col):
    """Load AUCell output and merge with cell type mapper."""
    anno = pd.read_csv(csv_path)
    anno = anno.merge(mapper, how='left', left_on='label', right_on='cell_type_short')
    return anno['cell_type_short'], anno['cell_type'], anno['cell_category_from_type']

# Reference (WTX ground-truth annotation)
ref_cat = adata_test.obs['aucell_cell_category_from_type'].tolist()

# GCN (our model, ywlb prediction)
gcn_st, gcn_ft, gcn_cat = load_aucell(
    f"{AUCELL_DIR}/half_log1p_half_ywlb_imp_aucell.csv",
    'gcn_cell_type_short', 'gcn_cell_type_full', 'gcn_cell_category')

# gimVI
gimvi_st, gimvi_ft, gimvi_cat = load_aucell(
    f"{AUCELL_DIR}/half_log1p_half_gimvi_imp_aucell.csv",
    'gimvi_cell_type_short', 'gimvi_cell_type_full', 'gimvi_cell_category')

# stPlus
stplus_st, stplus_ft, stplus_cat = load_aucell(
    f"{AUCELL_DIR}/half_log1p_half_stplus_imp_aucell.csv",
    'stplus_cell_type_short', 'stplus_cell_type_full', 'stplus_cell_category')

# Tangram
tangram_st, tangram_ft, tangram_cat = load_aucell(
    f"{AUCELL_DIR}/half_log1p_half_tangram_imp_aucell.csv",
    'tangram_cell_type_short', 'tangram_cell_type_full', 'tangram_cell_category')

# SPRITE (SPage + SPRITE spatial smoothing)
sprite_st, sprite_ft, sprite_cat = load_aucell(
    f"{AUCELL_DIR}/half_log1p_half_sprite_imp_aucell.csv",
    'sprite_cell_type_short', 'sprite_cell_type_full', 'sprite_cell_category')

# CellPLM
cellplm_st, cellplm_ft, cellplm_cat = load_aucell(
    f"{AUCELL_DIR}/half_log1p_half_cellplm_imp_aucell.csv",
    'cellplm_cell_type_short', 'cellplm_cell_type_full', 'cellplm_cell_category')

## 3. Classification metrics (Hamming loss and Micro F1)

In [ ]:
method_preds = {
    'GCN':     gcn_cat.tolist(),
    'Tangram': tangram_cat.tolist(),
    'gimVI':   gimvi_cat.tolist(),
    'stPlus':  stplus_cat.tolist(),
    'SPRITE':  sprite_cat.tolist(),
    'CellPLM': cellplm_cat.tolist(),
}

hamming = {m: hamming_loss(ref_cat, pred) for m, pred in method_preds.items()}
micro_f1 = {m: f1_score(ref_cat, pred, average='micro') for m, pred in method_preds.items()}

imp_res = pd.DataFrame({
    'Imputation': list(micro_f1.keys()),
    'Micro F1':   [round(v, 3) for v in micro_f1.values()],
    'Hamming':    [round(v, 3) for v in hamming.values()]
})
print(imp_res.to_string(index=False))

In [ ]:
# Formatted table figure (paper)
df_table = imp_res.set_index('Imputation').T

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')
col_widths = [0.18] * len(df_table.columns)
table = ax.table(cellText=df_table.values, rowLabels=df_table.index,
                 colLabels=df_table.columns, loc='center',
                 cellLoc='center', colWidths=col_widths)
table.scale(1.0, 2.2)
table.auto_set_font_size(False)
table.set_fontsize(13)

# Bold best values (GCN column: highest Micro F1, lowest Hamming)
best_col = df_table.columns.tolist().index('GCN')
for row_idx in range(len(df_table)):
    table[(row_idx + 1, best_col)].set_text_props(weight='bold')

plt.tight_layout()
plt.savefig("/path/to/output/imputation_classification_table.pdf", bbox_inches='tight', dpi=300)
plt.show()

## 4. Clustering quality across CosMx panels (Supplementary Table 6)

In [ ]:
# Load all four panel objects
adata_wtx        = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_merged_anno.h5ad")
adata_6k_custom  = sc.read_h5ad(f"{DATA_DIR}/qc_6k_merged_log1p_anno_6375genes.h5ad")
adata_6k         = sc.read_h5ad(f"{DATA_DIR}/qc_6k_merged_log1p_anno_6175genes.h5ad")
adata_1k         = sc.read_h5ad(f"{DATA_DIR}/imp_ywl_anno.h5ad")

In [ ]:
def clustering_metrics(adata, label_col, reduction):
    labels = adata.obs[label_col].tolist()
    emb    = adata.obsm[reduction]
    sil = silhouette_score(emb, labels)
    ch  = calinski_harabasz_score(emb, labels)
    db  = davies_bouldin_score(emb, labels)
    return round(sil, 3), round(ch, 2), round(db, 3)

# PCA-based
pca_wtx       = clustering_metrics(adata_wtx,       'aucell_cell_type_short',   'X_pca')
pca_6k_custom = clustering_metrics(adata_6k_custom, 'aucell_cell_type_short',   'X_pca')
pca_6k        = clustering_metrics(adata_6k,        'aucell_cell_type_short',   'X_pca')
pca_1k        = clustering_metrics(adata_1k,        'only_1k_cell_type_short',  'X_pca')
pca_1k_imp    = clustering_metrics(adata_1k,        'v7_ywl_b_imp_cell_type_short', 'X_pca')

# Harmony-based
harm_wtx       = clustering_metrics(adata_wtx,       'aucell_cell_type_short',   'X_pca_harmony')
harm_6k_custom = clustering_metrics(adata_6k_custom, 'aucell_cell_type_short',   'X_pca_harmony')
harm_6k        = clustering_metrics(adata_6k,        'aucell_cell_type_short',   'X_pca_harmony')
harm_1k        = clustering_metrics(adata_1k,        'only_1k_cell_type_short',  'X_pca_harmony')
harm_1k_imp    = clustering_metrics(adata_1k,        'v7_ywl_b_imp_cell_type_short', 'X_pca_harmony')

for label, pca, harm in zip(
    ['WTX', '6K+Custom', '6K', '1K (only)', '1K+GCN'],
    [pca_wtx, pca_6k_custom, pca_6k, pca_1k, pca_1k_imp],
    [harm_wtx, harm_6k_custom, harm_6k, harm_1k, harm_1k_imp]
):
    print(f"{label:14s}  PCA: sil={pca[0]:.3f} CH={pca[1]:.1f} DB={pca[2]:.3f}"
          f"  |  Harmony: sil={harm[0]:.3f} CH={harm[1]:.1f} DB={harm[2]:.3f}")

In [ ]:
# Format Silhouette + CH table (paper / Supp Table 6)
def make_table_figure(data_dict, title, save_path):
    df = pd.DataFrame(data_dict).set_index('Metric')
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.axis('off')
    col_widths = [0.2] * len(df.columns)
    table = ax.table(cellText=df.values, rowLabels=df.index,
                     colLabels=df.columns, loc='center',
                     cellLoc='center', colWidths=col_widths)
    table.scale(1.0, 2.2)
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    ax.set_title(title, fontsize=12, pad=20)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

pca_table = {
    'Metric':    ['Silhouette', 'CH Index', 'DB Index'],
    'WTX':       list(pca_wtx),
    '6K+Custom': list(pca_6k_custom),
    '6K':        list(pca_6k),
    '1K':        list(pca_1k),
    '1K+GCN':    list(pca_1k_imp),
}
make_table_figure(pca_table, 'Clustering quality (PCA)',
                  '/path/to/output/clustering_metrics_pca.pdf')

harm_table = {
    'Metric':    ['Silhouette', 'CH Index', 'DB Index'],
    'WTX':       list(harm_wtx),
    '6K+Custom': list(harm_6k_custom),
    '6K':        list(harm_6k),
    '1K':        list(harm_1k),
    '1K+GCN':    list(harm_1k_imp),
}
make_table_figure(harm_table, 'Clustering quality (Harmony)',
                  '/path/to/output/clustering_metrics_harmony.pdf')

## 5. Per-gene imputation correlation (PCC and SCC)

Pearson and Spearman correlations between imputed and true expression for 12 marker genes,
grouped by cell type category.

- **GCN**: pseudo-6K imputed counts (`half_ori_half_ywlb_imp_log1p.csv`)
- **Tangram / stPlus**: pseudo-1K imputed counts (see `02_tangram_imputation.ipynb` and `04_stplus_imputation.ipynb`)


In [ ]:
# Load ground truth (full transcriptome, test cells)
adata_norm = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_I0261_I0331_anno.h5ad")
adata_norm.obs['fov'] = adata_norm.obs['fov'].astype(int)

test_cells_norm = [
    cell for fov, cell in zip(adata_norm.obs['fov'], adata_norm.obs['fov_cell_id'])
    if not ((fov > 9 and fov <= 32) or (fov > 44))
]
test_adata_norm = adata_norm[adata_norm.obs['fov_cell_id'].isin(test_cells_norm), :].copy()
df_true = pd.DataFrame(test_adata_norm.X)
df_true.columns = test_adata_norm.var_names.tolist()
df_true.index   = test_adata_norm.obs['fov_cell_id'].tolist()

In [ ]:
# GCN: pseudo-6K imputed counts
gcn_count    = pd.read_csv(f"{COUNT_DIR}/half_ori_half_ywlb_imp_log1p.csv",   index_col=0)

# Tangram and stPlus: pseudo-1K imputed counts
tangram_count = pd.read_csv(f"{COUNT_DIR}/1k_ori_17k_tangram_imp_log1p.csv",  index_col=0)
stplus_count  = pd.read_csv(f"{COUNT_DIR}/1k_ori_17k_stplus_imp_log1p.csv",   index_col=0)

In [ ]:
# Marker genes grouped by cell type category
marker_genes = [
    'ATOH1', 'BEST4', 'OTOP2', 'REG4',          # Epithelial differentiation
    'SLC26A3', 'SLC5A8', 'AQP8', 'CA1',          # Epithelial transport
    'CEACAM7', 'CLCA1',                           # Goblet / secretory
    'CTHRC1', 'ASPN',                             # Stromal
]
gene_groups = [
    (0,  4,  'Epithelial\ndifferentiation'),
    (4,  8,  'Epithelial\ntransport'),
    (8,  10, 'Goblet /\nsecretory'),
    (10, 12, 'Stromal'),
]

methods = {
    'GCN':     gcn_count,
    'Tangram': tangram_count,
    'stPlus':  stplus_count,
}

corr_results = []
for gene in marker_genes:
    true_expr = df_true[gene]
    for method_name, df_imp in methods.items():
        pcc, _ = pearsonr(true_expr, df_imp[gene])
        scc, _ = spearmanr(true_expr, df_imp[gene])
        corr_results.append({'gene': gene, 'method': method_name, 'PCC': pcc, 'SCC': scc})

df_corr = pd.DataFrame(corr_results)
print(df_corr.pivot(index='gene', columns='method', values='PCC').round(3))

In [ ]:
# Grouped PCC heatmap (paper figure)
pcc_pivot = df_corr.pivot(index='gene', columns='method', values='PCC')
pcc_pivot = pcc_pivot.loc[marker_genes, ['GCN', 'Tangram', 'stPlus']]

fig, ax = plt.subplots(figsize=(5, 7))
sns.heatmap(pcc_pivot, annot=True, fmt='.3f', cmap='Reds',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)

for start, end, label in gene_groups:
    if start > 0:
        ax.axhline(y=start, color='black', linewidth=1)

ax.set_ylabel('')
ax.set_xlabel('')
ax.set_title('Per-Gene Imputation Accuracy (Pearson)', fontsize=11)

for start, end, label in gene_groups:
    mid = (start + end) / 2
    ax.text(-0.55, mid, label, ha='center', va='center',
            fontsize=10, transform=ax.get_yaxis_transform())

plt.subplots_adjust(left=0.35)
plt.savefig('/path/to/output/gene_imputation_pcc_grouped.pdf', dpi=300, bbox_inches='tight')
plt.show()